In [ ]:
!pip install gradio faiss-cpu pandas numpy sentence-transformers \
             transformers bitsandbytes accelerate torch -q
import os
import json
import re
import time
import zipfile
import shutil
import threading
import sys
import logging
import warnings
import requests                          # ← added for PROD download
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import faiss
import torch
import gradio as gr

from sentence_transformers import SentenceTransformer
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TextIteratorStreamer,
)

# ============================================================================
# Log / output clearing
# ============================================================================

def clear_output_logs():
    """
    Silence noisy loggers and wipe the cell/terminal output so that Gradio
    launches into a clean screen with no model-download chatter above it.

    Strategy (applied in order):
      1. Set all heavyweight library loggers to ERROR so nothing new prints.
      2. Try IPython clear_output() — works in Colab, Kaggle, Jupyter.
      3. Fall back to an ANSI escape sequence — works in any real terminal.
    """
    # ── 1. Silence verbose loggers ────────────────────────────────────────
    _noisy = [
        "transformers", "transformers.modeling_utils",
        "sentence_transformers", "gradio", "gradio.analytics",
        "httpx", "httpcore", "urllib3", "filelock",
        "huggingface_hub", "huggingface_hub.file_download",
        "faiss", "accelerate", "bitsandbytes",
    ]
    for name in _noisy:
        logging.getLogger(name).setLevel(logging.ERROR)

    # Suppress any future warnings globally
    warnings.filterwarnings("ignore")

    # ── 2. IPython / Colab / Kaggle ───────────────────────────────────────
    try:
        from IPython.display import clear_output as _ipy_clear
        _ipy_clear(wait=False)
        return   # Done — IPython handled it
    except ImportError:
        pass

    # ── 3. ANSI terminal fallback ─────────────────────────────────────────
    # \033[2J = erase entire screen, \033[H = move cursor to top-left
    sys.stdout.write("\033[2J\033[H")
    sys.stdout.flush()


# ============================================================================
# Global config
# ============================================================================
STAGE          = "PROD"   # "TEST" | "PROD"
GA_TRACKING_ID = "G-03K3WT21HP"   # ← updated

# ============================================================================
# Global caches & constants
# ============================================================================
_llm_model     = None
_llm_tokenizer = None
_embedder      = None
_rag_cache: dict = {}
_llm_ready     = False
_llm_loading   = False

KNOWLEDGEBASE_ROOT = "knowledgebase"
LLM_MODEL_ID       = "Qwen/Qwen2.5-7B-Instruct"
EMBED_MODEL_ID     = "sentence-transformers/all-MiniLM-L6-v2"
TOP_K              = 4
MAX_NEW_TOKENS     = 256
MAX_HISTORY_TURNS  = 6
TILES_PER_ROW      = 3

# ── Agentic RAG constants ──────────────────────────────────────────────────
# Maximum reasoning steps before forcing a final answer.
# Kept at 3 so worst-case GPU time stays within ~45 s on a T4.
MAX_AGENT_STEPS    = 3
# Hard cap on injected context characters to protect the context window.
MAX_CONTEXT_CHARS  = 3000

# ============================================================================
# Model loading
# ============================================================================

def load_embedder():
    global _embedder
    if _embedder is None:
        _embedder = SentenceTransformer(EMBED_MODEL_ID)
    return _embedder


def load_llm():
    global _llm_model, _llm_tokenizer, _llm_ready, _llm_loading
    if _llm_ready:
        return _llm_tokenizer, _llm_model
    _llm_loading = True
    try:
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
        )
        _llm_tokenizer = AutoTokenizer.from_pretrained(
            LLM_MODEL_ID, trust_remote_code=True
        )
        _llm_model = AutoModelForCausalLM.from_pretrained(
            LLM_MODEL_ID,
            quantization_config=bnb_config,
            device_map="auto",
            trust_remote_code=True,
        )
        _llm_model.eval()
        _llm_ready = True
    finally:
        _llm_loading = False
    return _llm_tokenizer, _llm_model


# Models are loaded in the foreground at entry point (see __main__ block below)
# so progress is visible in Colab / Kaggle cell output before Gradio starts.


# ============================================================================
# RAG helpers
# ============================================================================

def load_rag(board: str, cls: str, subject: str):
    key = f"{board}/{cls}/{subject}"
    if key not in _rag_cache:
        path  = os.path.join(KNOWLEDGEBASE_ROOT, board, cls, subject)
        index = faiss.read_index(os.path.join(path, "faiss_index.bin"))
        df    = pd.read_parquet(os.path.join(path, "metadata.parquet"))
        _rag_cache[key] = (index, df)
    return _rag_cache[key]


def retrieve_chunks(query: str, index, df: pd.DataFrame) -> str:
    embedder   = load_embedder()
    q_vec      = embedder.encode([query], normalize_embeddings=True).astype("float32")
    _, indices = index.search(q_vec, TOP_K)
    chunks = []
    for idx in indices[0]:
        if idx < len(df):
            row  = df.iloc[idx]
            text = row.get("text", row.get("chunk", row.get("content", "")))
            chunks.append(str(text).strip())
    return "\n\n".join(chunks)


# ============================================================================
# Filesystem helpers
# ============================================================================

def list_dirs(path: str) -> list:
    if not os.path.isdir(path):
        return []
    return sorted(
        d for d in os.listdir(path)
        if os.path.isdir(os.path.join(path, d)) and not d.startswith(".")
    )


def get_boards()             -> list: return list_dirs(KNOWLEDGEBASE_ROOT)
def get_classes(board)       -> list: return list_dirs(os.path.join(KNOWLEDGEBASE_ROOT, board))
def get_subjects(board, cls) -> list: return list_dirs(os.path.join(KNOWLEDGEBASE_ROOT, board, cls))


def get_nav_items(state: dict) -> list:
    level = state["level"]
    if level == "board":   return get_boards()
    if level == "class":   return get_classes(state["board"])
    if level == "subject": return get_subjects(state["board"], state["cls"])
    return []


def get_nav_title(state: dict) -> str:
    level = state["level"]
    if level == "board":   return "Choose your Board"
    if level == "class":   return f"Choose a Class  —  {state['board']}"
    if level == "subject": return f"Choose a Subject  —  {state['board']} › {state['cls']}"
    return ""


def get_breadcrumb(state: dict) -> str:
    parts = []
    if state.get("board"):   parts.append(state["board"])
    if state.get("cls"):     parts.append(state["cls"])
    if state.get("subject"): parts.append(state["subject"])
    return " › ".join(parts)


# ============================================================================
# Knowledgebase upload + validation
# ============================================================================

def _validate_knowledgebase(root: str):
    required    = {"faiss_index.bin", "metadata.parquet", "config.json"}
    valid_count = 0
    boards      = list_dirs(root)
    if not boards:
        return False, "No board folders found inside the uploaded archive."
    for board in boards:
        for cls in list_dirs(os.path.join(root, board)):
            for subject in list_dirs(os.path.join(root, board, cls)):
                subj_path = os.path.join(root, board, cls, subject)
                if required.issubset(set(os.listdir(subj_path))):
                    valid_count += 1
    if valid_count == 0:
        return False, (
            "Structure looks incorrect. Each subject folder must contain "
            "faiss_index.bin, metadata.parquet, and config.json."
        )
    return True, f"Knowledgebase validated. Found {valid_count} subject(s) ready to use."


def handle_kb_upload(uploaded_file, state: dict):
    if uploaded_file is None:
        return state, '<p class="status-err">No file received. Please upload a ZIP archive.</p>'
    try:
        if os.path.exists(KNOWLEDGEBASE_ROOT):
            shutil.rmtree(KNOWLEDGEBASE_ROOT)
        os.makedirs(KNOWLEDGEBASE_ROOT, exist_ok=True)

        file_path = uploaded_file if isinstance(uploaded_file, str) else uploaded_file.name
        if not zipfile.is_zipfile(file_path):
            return state, '<p class="status-err">The uploaded file is not a valid ZIP archive.</p>'

        with zipfile.ZipFile(file_path, "r") as zf:
            zf.extractall(KNOWLEDGEBASE_ROOT)

        top_level = list_dirs(KNOWLEDGEBASE_ROOT)
        if len(top_level) == 1 and top_level[0].lower() == "knowledgebase":
            inner = os.path.join(KNOWLEDGEBASE_ROOT, top_level[0])
            for item in os.listdir(inner):
                shutil.move(os.path.join(inner, item), KNOWLEDGEBASE_ROOT)
            shutil.rmtree(inner)

        valid, message = _validate_knowledgebase(KNOWLEDGEBASE_ROOT)
        if valid:
            new_state = {**state, "kb_loaded": True}
            return new_state, f'<p class="status-ok">{message}</p>'
        else:
            shutil.rmtree(KNOWLEDGEBASE_ROOT)
            os.makedirs(KNOWLEDGEBASE_ROOT, exist_ok=True)
            return state, f'<p class="status-err">{message}</p>'
    except Exception as exc:
        return state, f'<p class="status-err">Upload failed: {exc}</p>'


# ============================================================================
# NEW FUNCTION — Auto-download knowledgebase (PROD mode only)
# ============================================================================

_KB_DOWNLOAD_URL = (
    "https://raw.githubusercontent.com/buildwithsupratim/maria/main/knowledgebase.zip"
)
_KB_ZIP_PATH = "knowledgebase.zip"
_KB_TMP_DIR  = "_kb_tmp_extract"


def download_knowledgebase() -> None:
    """
    Download and extract the knowledgebase from GitHub (PROD mode).

    Steps:
      1. Skip if KNOWLEDGEBASE_ROOT already exists and is non-empty (cache guard).
      2. Stream-download the ZIP via requests (Colab-safe, no wget/curl).
      3. Validate the downloaded file is a real ZIP.
      4. Extract into a temporary directory first.
      5. Promote the contents into KNOWLEDGEBASE_ROOT, handling the case where
         the ZIP contains a top-level 'knowledgebase/' folder.
      6. Validate structure via _validate_knowledgebase().
      7. Clean up the ZIP and temp directory regardless of outcome.
    """
    # ── 1. Cache guard ─────────────────────────────────────────────────────
    if os.path.exists(KNOWLEDGEBASE_ROOT) and os.listdir(KNOWLEDGEBASE_ROOT):
        print("Knowledgebase already exists. Skipping download.")
        return

    # ── 2. Stream download ─────────────────────────────────────────────────
    print("Downloading knowledgebase...")
    try:
        with requests.get(_KB_DOWNLOAD_URL, stream=True, timeout=120) as r:
            r.raise_for_status()
            with open(_KB_ZIP_PATH, "wb") as f:
                for chunk in r.iter_content(chunk_size=8192):
                    if chunk:
                        f.write(chunk)
    except requests.RequestException as exc:
        # Clean up any partial file before surfacing the error.
        if os.path.exists(_KB_ZIP_PATH):
            os.remove(_KB_ZIP_PATH)
        raise RuntimeError(
            f"Network failure while downloading knowledgebase: {exc}"
        ) from exc

    # ── 3. Validate ZIP integrity ──────────────────────────────────────────
    if not zipfile.is_zipfile(_KB_ZIP_PATH):
        os.remove(_KB_ZIP_PATH)
        raise RuntimeError(
            "Downloaded file is not a valid ZIP archive. "
            "The remote file may be corrupt or the URL may have changed."
        )

    # ── 4. Extract into a temp directory ──────────────────────────────────
    print("Extracting knowledgebase...")
    if os.path.exists(_KB_TMP_DIR):
        shutil.rmtree(_KB_TMP_DIR)
    os.makedirs(_KB_TMP_DIR, exist_ok=True)

    try:
        with zipfile.ZipFile(_KB_ZIP_PATH, "r") as zf:
            zf.extractall(_KB_TMP_DIR)
    except zipfile.BadZipFile as exc:
        shutil.rmtree(_KB_TMP_DIR, ignore_errors=True)
        os.remove(_KB_ZIP_PATH)
        raise RuntimeError(
            f"Corrupt ZIP archive — extraction failed: {exc}"
        ) from exc

    # ── 5. Promote contents into KNOWLEDGEBASE_ROOT ────────────────────────
    # Handle two possible ZIP layouts:
    #   A) knowledgebase.zip → board/class/subject/...       (flat)
    #   B) knowledgebase.zip → knowledgebase/board/class/... (wrapped)
    os.makedirs(KNOWLEDGEBASE_ROOT, exist_ok=True)

    top_level = list_dirs(_KB_TMP_DIR)
    if len(top_level) == 1 and top_level[0].lower() == "knowledgebase":
        # Layout B — unwrap the inner folder
        inner_dir = os.path.join(_KB_TMP_DIR, top_level[0])
        for item in os.listdir(inner_dir):
            shutil.move(os.path.join(inner_dir, item), KNOWLEDGEBASE_ROOT)
    else:
        # Layout A — move everything directly
        for item in os.listdir(_KB_TMP_DIR):
            shutil.move(os.path.join(_KB_TMP_DIR, item), KNOWLEDGEBASE_ROOT)

    # ── 6. Validate structure ──────────────────────────────────────────────
    valid, message = _validate_knowledgebase(KNOWLEDGEBASE_ROOT)
    if not valid:
        shutil.rmtree(KNOWLEDGEBASE_ROOT, ignore_errors=True)
        raise RuntimeError(
            f"Knowledgebase validation failed after extraction: {message}"
        )

    # ── 7. Clean up ────────────────────────────────────────────────────────
    shutil.rmtree(_KB_TMP_DIR, ignore_errors=True)
    if os.path.exists(_KB_ZIP_PATH):
        os.remove(_KB_ZIP_PATH)

    print("Knowledgebase ready.")


# ============================================================================
# ── AGENTIC RAG ─────────────────────────────────────────────────────────────
# ============================================================================

# ── Original prompt kept for backward-compat reference (not used in agent) ─
SYSTEM_PROMPT = (
    "You are a warm, loving, and patient teacher who cares deeply about every child you teach. "
    "You have the heart of a child psychologist, the knowledge of a skilled teacher, and the "
    "creativity of a storyteller. Your tone is always gentle, encouraging, and joyful. "
    "You speak simply and clearly. You never rush, never scold, and always make the child feel "
    "safe and capable. You use the provided context to answer questions accurately and never make "
    "things up. Keep your responses concise, warm, and easy to understand."
)

# ── Agent system prompt ────────────────────────────────────────────────────
AGENT_SYSTEM_PROMPT = """You are a warm, loving, and patient teacher with the heart of a child psychologist and the creativity of a storyteller.
You are also a careful thinker who knows when to look things up and when to answer from what you already know.

You must respond with a single JSON object — nothing else before or after it.

You have two available actions:

1. SEARCH — use this when you need facts, definitions, or curriculum content to give an accurate answer.
   Return exactly:
   {"action": "search", "query": "<a short, specific search query>"}

2. ANSWER — use this when you are confident and ready to speak to the child.
   Return exactly:
   {"action": "answer", "response": "<your warm, child-friendly response>"}

Rules you must always follow:
- Always prefer SEARCH before answering any academic or factual question.
- Never call SEARCH more than twice in one conversation turn.
- If you have already searched and have context, use ANSWER.
- Never make up facts. If you are unsure and cannot search, say so gently.
- Your ANSWER must be warm, simple, encouraging, and age-appropriate.
- Never use emojis. Never sound like a textbook or a robot.
- Keep answers concise — 3 to 6 sentences is ideal.
- If a child seems confused or sad, acknowledge that feeling first.
- Celebrate effort, not just correct answers.
"""


# ── Tool — search the knowledgebase ───────────────────────────────────────
def search_knowledgebase(query: str, rag_state: dict) -> str:
    """
    Thin wrapper around existing load_rag() + retrieve_chunks().
    Returns retrieved context string, or empty string if RAG is unavailable.
    """
    if not rag_state.get("loaded"):
        return ""
    try:
        index, df = load_rag(rag_state["board"], rag_state["cls"], rag_state["subject"])
        return retrieve_chunks(query, index, df)
    except Exception:
        return ""


# ── Action parser ──────────────────────────────────────────────────────────
def parse_agent_output(text: str) -> dict:
    """
    Attempt to extract a JSON action dict from the model output.

    Tries in order:
      1. Direct json.loads on the full text.
      2. Regex extraction of the first {...} block (handles extra prose).
      3. Fallback: treat the whole text as a plain answer.

    Always returns a dict with an "action" key.
    """
    text = text.strip()

    # Attempt 1 — clean JSON
    try:
        result = json.loads(text)
        if "action" in result:
            return result
    except (json.JSONDecodeError, ValueError):
        pass

    # Attempt 2 — extract first JSON object from surrounding text
    match = re.search(r"\{[^{}]+\}", text, re.DOTALL)
    if match:
        try:
            result = json.loads(match.group())
            if "action" in result:
                return result
        except (json.JSONDecodeError, ValueError):
            pass

    # Attempt 3 — fallback: the model gave prose, treat as a direct answer
    return {"action": "answer", "response": text}


# ── Agent prompt builder ───────────────────────────────────────────────────
def build_agent_prompt(
    context: str,
    scratchpad: str,
    conversation: list,
    user_message: str,
) -> list:
    """
    Assemble the message list for one agent reasoning step.
    """
    messages = [{"role": "system", "content": AGENT_SYSTEM_PROMPT}]

    # Inject retrieved context if we have any
    if context:
        truncated = context[:MAX_CONTEXT_CHARS]
        if len(context) > MAX_CONTEXT_CHARS:
            truncated += "\n[...context truncated for length...]"
        messages.append({
            "role": "system",
            "content": (
                "[RETRIEVED KNOWLEDGE — use this to inform your answer]\n"
                f"{truncated}\n"
                "[END OF RETRIEVED KNOWLEDGE]"
            ),
        })

    # Inject scratchpad so the model knows what it has already done
    if scratchpad:
        messages.append({
            "role": "system",
            "content": (
                "[ACTIONS TAKEN SO FAR THIS TURN]\n"
                f"{scratchpad}\n"
                "[END OF ACTIONS]"
            ),
        })

    # Trimmed conversation history
    trimmed = conversation[-(MAX_HISTORY_TURNS * 2):]
    for turn in trimmed:
        messages.append({"role": turn["role"], "content": turn["content"]})

    messages.append({"role": "user", "content": user_message})
    return messages


# ── ORIGINAL build_prompt kept for reference (no longer called) ─────────────
def build_prompt(context: str, conversation: list, user_message: str) -> list:
    """Original single-pass prompt builder — preserved, not used by agent."""
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    if context:
        messages.append({
            "role": "system",
            "content": (
                f"[CONTEXT]\n{context}\n[END CONTEXT]\n\n"
                "Use the context above to help answer the child's question."
            ),
        })
    trimmed = conversation[-(MAX_HISTORY_TURNS * 2):]
    for turn in trimmed:
        messages.append({"role": turn["role"], "content": turn["content"]})
    messages.append({"role": "user", "content": user_message})
    return messages


# ── Non-streaming single generation step (used for SEARCH decisions) ────────
def _run_model_greedy(messages: list, tokenizer, model, max_tokens: int = 128) -> str:
    """
    Run a fast, non-streamed generation pass for intermediate reasoning steps.
    Uses greedy decoding (do_sample=False) for determinism and speed.
    Max tokens is kept small because we only need a short JSON decision.
    """
    text   = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([text], return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    generated = output[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()


# ── generate_response — agentic loop ──────────────────────────────────────
def generate_response(user_message: str, conversation: list, rag_state: dict):
    """
    Agentic RAG loop.

    Flow per turn (max MAX_AGENT_STEPS iterations):
      Step N:
        1. Build agent prompt with accumulated context + scratchpad.
        2. Run model (greedy, non-streamed) to get an action JSON.
        3. Parse action:
           a. "search"  → call search_knowledgebase(), append to context +
                          scratchpad, continue loop.
           b. "answer"  → extract the clean "response" text from the JSON
                          and stream it word-by-word to the UI.
                          *** FIX: we do NOT re-run the model here.
                          Re-running with the same JSON-output prompt was the
                          cause of raw JSON leaking into the chat window. ***
        4. Guard: if the same query is repeated → force "answer".
        5. Guard: if loop exhausted → yield last extracted response text.

    Only the final ANSWER step is streamed to the user.
    Search decisions are fast greedy passes invisible to the user.
    """
    if not _llm_ready:
        yield (
            "I am just getting ready — the model is still loading. "
            "Please try again in a moment."
        )
        return

    tokenizer, model = load_llm()

    # ── Agent state for this turn ──────────────────────────────────────────
    context    = ""
    scratchpad = ""
    seen_queries: set = set()
    last_raw   = ""

    for step in range(MAX_AGENT_STEPS):

        # ── Step 1: build prompt ─────────────────────────────────────────
        messages = build_agent_prompt(context, scratchpad, conversation, user_message)

        # ── Step 2: run model (non-streamed decision step) ───────────────
        raw_output = _run_model_greedy(messages, tokenizer, model, max_tokens=160)
        last_raw   = raw_output

        # ── Step 3: parse action ─────────────────────────────────────────
        action = parse_agent_output(raw_output)

        if action["action"] == "search":
            query = action.get("query", user_message).strip()

            if query in seen_queries or not rag_state.get("loaded"):
                scratchpad += (
                    f"\nStep {step+1}: Attempted SEARCH for '{query}' — "
                    "skipped (already done or RAG unavailable). Will answer now."
                )
                context += "\n[No additional search results — please answer based on what you know.]"
                continue

            seen_queries.add(query)
            retrieved = search_knowledgebase(query, rag_state)

            if retrieved:
                new_chunks = [
                    chunk for chunk in retrieved.split("\n\n")
                    if chunk.strip() and chunk.strip() not in context
                ]
                new_text = "\n\n".join(new_chunks)
                context += ("\n\n" + new_text) if context else new_text
                scratchpad += f"\nStep {step+1}: Searched for '{query}' — found {len(new_chunks)} new chunk(s)."
            else:
                scratchpad += f"\nStep {step+1}: Searched for '{query}' — no results found."

            continue

        elif action["action"] == "answer":
            response_text = action.get("response", "").strip()

            if not response_text:
                # Model returned an empty "response" field — use raw text as fallback
                # but make one more attempt to strip any JSON wrapper
                fallback = parse_agent_output(raw_output).get("response", raw_output)
                response_text = fallback.strip() or raw_output

            # ── FIX: stream the already-extracted plain-text response ────
            # Previously the code re-ran the model with streaming enabled,
            # but the same AGENT_SYSTEM_PROMPT instructs the model to output
            # JSON, so the stream contained raw JSON visible to the child.
            #
            # Solution: we already have `response_text` (the clean value of
            # the "response" key).  We simulate token streaming by yielding
            # it word-by-word, which keeps the live-typing feel in the UI
            # without any risk of JSON leaking through.
            words = response_text.split(" ")
            partial = ""
            for i, word in enumerate(words):
                partial += ("" if i == 0 else " ") + word
                yield partial
                time.sleep(0.03)   # ~30 ms per word — comfortable reading pace

            return  # Done — exit generator after streaming

        else:
            last_raw = raw_output
            break

    # ── Fallback: loop exhausted without an explicit ANSWER ────────────────
    if last_raw:
        fallback = parse_agent_output(last_raw).get("response", last_raw)
        yield fallback.strip() or last_raw
    else:
        yield (
            "I am sorry, I got a little confused there. "
            "Could you ask me that again in a different way?"
        )


# ============================================================================
# Chatbot history helper
# ============================================================================

def _build_chatbot_history(conv: list) -> list:
    """Convert flat {role, content} list → Gradio (user_msg, bot_msg) tuple list."""
    history = []
    i = 0
    while i < len(conv):
        if conv[i]["role"] == "assistant" and i == 0:
            history.append((None, conv[i]["content"]))
            i += 1
        elif conv[i]["role"] == "user":
            user_turn = conv[i]["content"]
            asst_turn = conv[i + 1]["content"] if i + 1 < len(conv) else None
            history.append((user_turn, asst_turn))
            i += 2
        else:
            i += 1
    return history


# ============================================================================
# CSS
# ============================================================================

CSS = """
/* ── Navigation tiles ───────────────────────────────────────────────────── */
.tile-btn {
    min-height: 72px !important;
    font-size: 0.97rem !important;
    font-weight: 600 !important;
    border-radius: 14px !important;
    background: linear-gradient(135deg,#e8f4f8 0%,#d0eaf5 100%) !important;
    border: 1.5px solid #a8d8ea !important;
    color: #1a4a5a !important;
    transition: transform 0.15s ease, box-shadow 0.15s ease !important;
    white-space: normal !important;
    line-height: 1.3 !important;
}
.tile-btn:hover {
    background: linear-gradient(135deg,#d0eaf5 0%,#b4d9ef 100%) !important;
    transform: translateY(-3px) !important;
    box-shadow: 0 6px 18px rgba(0,0,0,0.13) !important;
}
.tile-btn:active { transform: translateY(0) !important; box-shadow: none !important; }

/* ── Back button ─────────────────────────────────────────────────────────── */
.back-btn {
    background: #f0ecff !important;
    border: 1.5px solid #c0aef0 !important;
    color: #4a2a8a !important;
    border-radius: 10px !important;
    font-weight: 600 !important;
}
.back-btn:hover { background: #e2dbff !important; }

/* ── Upload panel ────────────────────────────────────────────────────────── */
.upload-panel {
    border: 2px dashed #a8d8ea !important;
    border-radius: 16px !important;
    padding: 22px !important;
    background: #f4fbff !important;
    margin-bottom: 22px !important;
}

/* ── Status / breadcrumb text ────────────────────────────────────────────── */
.status-ok  { color: #2e7d32; font-weight: 600; margin: 4px 0; }
.status-err { color: #c62828; font-weight: 600; margin: 4px 0; }
.breadcrumb-text { font-size:0.88rem !important; color:#6a8a9a !important; margin-bottom:4px !important; }
.nav-title  { font-size:1.25rem !important; color:#1a4a5a !important; font-weight:700 !important; margin-bottom:14px !important; }
.empty-state  { color:#9aabba !important; font-style:italic !important; padding:28px 0 !important; text-align:center !important; }
.lock-notice  { border:1.5px solid #ffe082 !important; background:#fffde7 !important; border-radius:12px !important; padding:16px 20px !important; color:#795548 !important; font-weight:500 !important; }

/* ── Top full-width banner video ─────────────────────────────────────────── */
/* Thin 5 : 1 cinematic banner — purely decorative, zero interaction.        */
.top-video-wrapper {
    width: 100% !important;
    position: relative !important;
    padding-bottom: 20% !important;      /* 5 : 1  → banner proportions      */
    height: 0 !important;
    overflow: hidden !important;
    border-radius: 0 !important;         /* edge-to-edge, no rounding         */
    margin: 0 0 0 0 !important;
    background: #000 !important;
    /* Hard block on all pointer / keyboard / selection interaction */
    pointer-events: none !important;
    user-select: none !important;
    -webkit-user-select: none !important;
}
.top-video-wrapper iframe {
    position: absolute !important;
    /* Pull the iframe up so the Vimeo letterbox is cropped by the container  */
    top: 50% !important;
    left: 0 !important;
    width: 100% !important;
    height: 300% !important;             /* oversized → fills 5:1 crop        */
    transform: translateY(-50%) !important;
    border: none !important;
    pointer-events: none !important;     /* belt-and-suspenders: no clicks     */
}

/* ── Feedback link bar ───────────────────────────────────────────────────── */
.feedback-bar {
    text-align: center !important;
    padding: 8px 0 14px !important;
}
.feedback-bar a {
    display: inline-block !important;
    color: #1a5a7a !important;
    font-weight: 600 !important;
    font-size: 0.9rem !important;
    text-decoration: none !important;
    background: #e8f4f8 !important;
    padding: 6px 20px !important;
    border-radius: 20px !important;
    border: 1px solid #a8d8ea !important;
    transition: background 0.15s ease !important;
}
.feedback-bar a:hover {
    background: #d0eaf5 !important;
    text-decoration: underline !important;
}
"""

# ── Top full-width Vimeo video ─────────────────────────────────────────────
# Vimeo embed parameters:
#   autoplay=1   → start playing immediately (muted so browser allows it)
#   muted=1      → required for autoplay without a prior user gesture
#   loop=1       → loop continuously
#   title=0      → hide Vimeo title overlay
#   byline=0     → hide author byline
#   portrait=0   → hide Vimeo logo
#
# The companion <script> block (MUTE_JS_HTML) listens for the player "ready"
# event and then:
#   • unmutes the video immediately (first visit / nav screens)
#   • re-mutes it whenever the chat panel becomes visible
#   • re-unmutes it whenever the user navigates back from chat
TOP_VIDEO_HTML = (
    '<div class="top-video-wrapper">'
    '<iframe'
    ' id="top-vimeo-player"'
    ' src="https://player.vimeo.com/video/1178060711'
    # autoplay+muted  → browser allows autoplay without user gesture
    # controls=0      → hides play/pause bar so it is truly non-interactive
    # loop=1          → continuous loop
    # title/byline/portrait=0 → hide all Vimeo UI chrome
    '?autoplay=1&muted=1&loop=1&controls=0&title=0&byline=0&portrait=0"'
    ' allow="autoplay; picture-in-picture"'
    ' tabindex="-1"'           # remove from keyboard tab order
    ' aria-hidden="true"'      # invisible to screen readers
    '>'
    '</iframe>'
    '</div>'
)

# ── Feedback link ──────────────────────────────────────────────────────────
FEEDBACK_HTML = (
    '<div class="feedback-bar">'
    '<a href="https://maria.canny.io/feedback" target="_blank" rel="noopener noreferrer">'
    '💬 Share your feedback'
    '</a>'
    '</div>'
)

# ── Google Analytics ───────────────────────────────────────────────────────
GA_HTML = (
    f'<!-- Google tag (gtag.js) -->\n'
    f'<script async src="https://www.googletagmanager.com/gtag/js?id={GA_TRACKING_ID}"></script>\n'
    "<script>\n"
    "  window.dataLayer = window.dataLayer || [];\n"
    "  function gtag(){dataLayer.push(arguments);}\n"
    "  gtag('js', new Date());\n"
    f"  gtag('config', '{GA_TRACKING_ID}');\n"
    "</script>"
)

# ── Mute / unmute JS ───────────────────────────────────────────────────────
# Uses the Vimeo postMessage API (no external SDK needed).
#
# Two complementary strategies ensure mute fires exactly when a subject is
# selected, with zero perceptible delay:
#
#   A) IMMEDIATE — a delegated 'click' listener on document fires for any
#      element that carries the CSS class 'subject-tile' (added by draw_tiles
#      when level == "subject").  This runs synchronously inside the click
#      event, so the video is muted the instant the child taps a subject.
#
#   B) FALLBACK — a MutationObserver watches #chat_panel_col for visibility
#      changes.  This catches any edge-case where the tile-click path is
#      bypassed (e.g. programmatic navigation) and also handles un-muting
#      when the user returns from chat to the nav screens.
MUTE_JS_HTML = """
<script>
(function () {
    var VIMEO_ORIGIN = 'https://player.vimeo.com';
    var playerReady  = false;
    var chatVisible  = false;

    /* ── send a postMessage command to the Vimeo iframe ── */
    function vimeoCmd(method, value) {
        var iframe = document.getElementById('top-vimeo-player');
        if (!iframe || !iframe.contentWindow) return;
        var msg = JSON.stringify(
            value !== undefined ? { method: method, value: value } : { method: method }
        );
        iframe.contentWindow.postMessage(msg, VIMEO_ORIGIN);
    }

    function mutePlayer()   { vimeoCmd('setVolume', 0); }
    function unmutePlayer() { vimeoCmd('setVolume', 1); }

    /* ── handle ready / error events from the Vimeo player ── */
    window.addEventListener('message', function (e) {
        if (e.origin !== VIMEO_ORIGIN) return;
        var data;
        try { data = JSON.parse(e.data); } catch (_) { return; }

        if (data.event === 'ready') {
            playerReady = true;
            /* Apply the correct volume state as soon as the player is ready */
            if (chatVisible) mutePlayer();
            else             unmutePlayer();
        }
    });

    /* ── Strategy A: immediate mute on subject-tile click ── */
    /*
     * draw_tiles() adds the CSS class 'subject-tile' to every button rendered
     * at the subject selection level.  We use event delegation on document so
     * the listener works even though Gradio re-renders the tile buttons
     * dynamically via @gr.render.
     */
    document.addEventListener('click', function (e) {
        var target = e.target;
        /* Walk up to 3 levels to find the button (Gradio may wrap in spans) */
        for (var i = 0; i < 3; i++) {
            if (!target) break;
            if (target.classList && target.classList.contains('subject-tile')) {
                /* Mute immediately — before Python even responds */
                chatVisible = true;
                if (playerReady) mutePlayer();
                return;
            }
            target = target.parentElement;
        }
    }, true);   /* useCapture=true → fires before any other listener */

    /* ── Strategy B: MutationObserver fallback ── */
    function watchChatPanel() {
        var chatCol = document.getElementById('chat_panel_col');
        if (!chatCol) {
            setTimeout(watchChatPanel, 600);
            return;
        }

        function syncVolume() {
            /* Gradio hides panels via display:none */
            var nowVisible =
                window.getComputedStyle(chatCol).display !== 'none' &&
                chatCol.offsetParent !== null;

            if (nowVisible === chatVisible) return;   /* no state change */
            chatVisible = nowVisible;

            if (!playerReady) return;   /* ready event will apply state later */
            if (chatVisible) mutePlayer();
            else             unmutePlayer();
        }

        var mo = new MutationObserver(syncVolume);

        /* Watch the chat column itself */
        mo.observe(chatCol, {
            attributes: true,
            attributeFilter: ['style', 'class']
        });

        /* Also watch the immediate parent — Gradio sometimes toggles display
           on a wrapper element rather than the column itself */
        if (chatCol.parentElement) {
            mo.observe(chatCol.parentElement, {
                attributes: true,
                attributeFilter: ['style', 'class'],
                childList: false
            });
        }

        syncVolume();   /* run once on init */
    }

    /* Start watching once the DOM has settled */
    if (document.readyState === 'loading') {
        document.addEventListener('DOMContentLoaded', function () {
            setTimeout(watchChatPanel, 1000);
        });
    } else {
        setTimeout(watchChatPanel, 1000);
    }
})();
</script>
"""


# ============================================================================
# App
# ============================================================================

def build_app():

    init_state = {
        "level":      "board",
        "board":      None,
        "cls":        None,
        "subject":    None,
        "rag_loaded": False,
        "kb_loaded":  (STAGE == "PROD"),
    }

    with gr.Blocks(css=CSS, title="Learning Companion") as demo:

        # ── Shared state ─────────────────────────────────────────────────── #
        app_state  = gr.State(value=init_state)
        conv_state = gr.State(value=[])

        # ── Analytics (PROD only) ─────────────────────────────────────────── #
        if STAGE == "PROD":
            gr.HTML(GA_HTML)

        # ── Top full-width Vimeo video ────────────────────────────────────── #
        # Rendered at the very top so it spans the full content width above
        # all other UI elements.  The companion MUTE_JS_HTML (injected at the
        # bottom) controls volume: ON for nav screens, OFF in chat window.
        gr.HTML(TOP_VIDEO_HTML)

        # ── Feedback link (below video) ───────────────────────────────────── #
        gr.HTML(FEEDBACK_HTML)

        # ── Header ───────────────────────────────────────────────────────── #
        gr.Markdown(
            "## Welcome to Your Learning Companion\n"
            "*A warm, joyful place to explore and grow*"
        )
        status_md = gr.Markdown("")

        # ── Upload panel (TEST only) ──────────────────────────────────────── #
        with gr.Column(visible=(STAGE == "TEST"), elem_classes=["upload-panel"]) as upload_panel:
            gr.Markdown("### Upload Knowledgebase")
            gr.Markdown(
                "Upload a **ZIP file** of your knowledgebase before using the app.  \n"
                "Required layout: `Board / Class_N / Subject / "
                "faiss_index.bin + metadata.parquet + config.json`"
            )
            with gr.Row():
                kb_upload  = gr.File(
                    label="Upload Knowledgebase (ZIP)",
                    file_types=[".zip"],
                    file_count="single",
                    scale=5,
                )
                upload_btn = gr.Button("Validate & Load", variant="primary", scale=1)
            upload_status = gr.HTML("")

        # ── Chat panel ────────────────────────────────────────────────────── #
        # elem_id="chat_panel_col" is targeted by MUTE_JS_HTML to detect when
        # the user enters or leaves the chat window and mute / unmute the video.
        with gr.Column(visible=False, elem_id="chat_panel_col") as chat_panel:
            chat_breadcrumb_md = gr.Markdown("", elem_classes=["breadcrumb-text"])
            chatbot = gr.Chatbot(
                label="",
                height=480,
                bubble_full_width=False,
                show_label=False,
            )
            with gr.Row():
                chat_input = gr.Textbox(
                    placeholder="Type your message here and press Enter...",
                    show_label=False,
                    scale=8,
                    container=False,
                )
                send_btn = gr.Button("Send", scale=1, variant="primary")
            chat_back_btn = gr.Button(
                "Choose a Different Subject", elem_classes=["back-btn"]
            )

        # ── Navigation panel ──────────────────────────────────────────────── #
        # The background video has been removed from here; the full-width video
        # now lives at the top of the page (see TOP_VIDEO_HTML above).
        with gr.Column(visible=True, elem_id="nav_panel_col") as nav_panel:

            breadcrumb_md = gr.Markdown("", elem_classes=["breadcrumb-text"])
            nav_title_md  = gr.Markdown("", elem_classes=["nav-title"])

            @gr.render(inputs=[app_state])
            def draw_tiles(state):
                level = state.get("level", "board")

                if STAGE == "TEST" and not state.get("kb_loaded", False):
                    gr.HTML(
                        '<div class="lock-notice">'
                        "Please upload your knowledgebase above to begin."
                        "</div>"
                    )
                    return

                if level == "chat":
                    return

                items = get_nav_items(state)

                if not items:
                    gr.HTML('<p class="empty-state">No items found here.</p>')
                    return

                for row_start in range(0, len(items), TILES_PER_ROW):
                    chunk = items[row_start : row_start + TILES_PER_ROW]
                    with gr.Row():
                        for label in chunk:
                            # Add 'subject-tile' when at subject level so the
                            # JS click listener can mute the video immediately.
                            extra = ["subject-tile"] if level == "subject" else []
                            btn = gr.Button(label, elem_classes=["tile-btn"] + extra)
                            btn.click(
                                fn=on_tile_click,
                                inputs=[btn, app_state, conv_state],
                                outputs=[
                                    app_state,
                                    conv_state,
                                    nav_panel,
                                    chat_panel,
                                    status_md,
                                    breadcrumb_md,
                                    nav_title_md,
                                    back_btn,
                                    chatbot,
                                    chat_breadcrumb_md,
                                ],
                            )

            back_btn = gr.Button("Back", elem_classes=["back-btn"], visible=False)

        # ── Mute / unmute JS (injected after all panels are in the DOM) ───── #
        gr.HTML(MUTE_JS_HTML)

        # ── Tile click handler ────────────────────────────────────────────── #
        def on_tile_click(tile_label: str, state: dict, conv: list):

            if STAGE == "TEST" and not state.get("kb_loaded", False):
                return (
                    state, conv,
                    gr.update(), gr.update(), "",
                    gr.update(value=get_breadcrumb(state)),
                    gr.update(value=get_nav_title(state)),
                    gr.update(visible=False),
                    gr.update(),
                    gr.update(),
                )

            new_state = {**state}
            level     = state["level"]

            if level == "board":
                new_state.update({"board": tile_label, "level": "class"})
                return (
                    new_state, conv,
                    gr.update(visible=True),
                    gr.update(visible=False),
                    "",
                    gr.update(value=get_breadcrumb(new_state)),
                    gr.update(value=get_nav_title(new_state)),
                    gr.update(visible=False),
                    gr.update(),
                    gr.update(),
                )

            elif level == "class":
                new_state.update({"cls": tile_label, "level": "subject"})
                return (
                    new_state, conv,
                    gr.update(visible=True),
                    gr.update(visible=False),
                    "",
                    gr.update(value=get_breadcrumb(new_state)),
                    gr.update(value=get_nav_title(new_state)),
                    gr.update(visible=True),
                    gr.update(),
                    gr.update(),
                )

            elif level == "subject":
                new_state.update({
                    "subject":    tile_label,
                    "level":      "chat",
                    "rag_loaded": False,
                })
                greeting = (
                    "Hello there! I am so glad you are here today.\n\n"
                    f"We are going to explore {tile_label} together, and I promise "
                    "it will be a lovely adventure.\n\n"
                    "Before we begin, may I know your name? "
                    "And is there anything in particular you feel curious about today?"
                )
                new_conv  = [{"role": "assistant", "content": greeting}]
                chat_hist = _build_chatbot_history(new_conv)
                bc_text   = get_breadcrumb(new_state)
                return (
                    new_state,
                    new_conv,
                    gr.update(visible=False),   # hide nav
                    gr.update(visible=True),    # show chat  ← JS detects this → mutes video
                    "",
                    gr.update(value=bc_text),
                    gr.update(value=""),
                    gr.update(visible=False),
                    chat_hist,
                    gr.update(value=bc_text),
                )

            return (
                new_state, conv,
                gr.update(), gr.update(), "",
                gr.update(value=get_breadcrumb(new_state)),
                gr.update(value=get_nav_title(new_state)),
                gr.update(visible=False),
                gr.update(),
                gr.update(),
            )

        # ── Back button ───────────────────────────────────────────────────── #
        def on_back(state: dict):
            new_state = {**state}
            level     = state["level"]
            if level == "class":
                new_state.update({"level": "board", "board": None})
            elif level == "subject":
                new_state.update({"level": "class", "cls": None})
            elif level == "chat":
                new_state.update({"level": "subject", "subject": None})

            show_back = new_state["level"] not in ("board", "chat")
            return (
                new_state,
                gr.update(value=get_breadcrumb(new_state)),
                gr.update(value=get_nav_title(new_state)),
                gr.update(visible=show_back),
            )

        back_btn.click(
            fn=on_back,
            inputs=[app_state],
            outputs=[app_state, breadcrumb_md, nav_title_md, back_btn],
        )

        # ── Chat back button ──────────────────────────────────────────────── #
        def on_chat_back(state: dict):
            new_state = {**state, "level": "subject", "subject": None}
            return (
                new_state,
                gr.update(visible=True),   # show nav  ← JS detects chat hidden → unmutes
                gr.update(visible=False),  # hide chat
                [],
                gr.update(value=get_breadcrumb(new_state)),
                gr.update(value=get_nav_title(new_state)),
                gr.update(visible=True),
            )

        chat_back_btn.click(
            fn=on_chat_back,
            inputs=[app_state],
            outputs=[
                app_state,
                nav_panel, chat_panel, chatbot,
                breadcrumb_md, nav_title_md, back_btn,
            ],
        )

        # ── Upload handler ────────────────────────────────────────────────── #
        def on_upload(uploaded_file, state: dict):
            new_state, status_html = handle_kb_upload(uploaded_file, state)
            return new_state, status_html

        upload_btn.click(
            fn=on_upload,
            inputs=[kb_upload, app_state],
            outputs=[app_state, upload_status],
        )

        # ── Chat send — streaming via agent loop ──────────────────────────── #
        def on_send(user_msg: str, conv: list, state: dict, history):
            if not user_msg.strip():
                yield conv, history, ""
                return

            rag_state = {
                "loaded":  state["level"] == "chat" and bool(state.get("subject")),
                "board":   state.get("board"),
                "cls":     state.get("cls"),
                "subject": state.get("subject"),
            }

            conv = list(conv)
            conv.append({"role": "user",      "content": user_msg})
            conv.append({"role": "assistant", "content": ""})

            for partial in generate_response(user_msg, conv[:-2], rag_state):
                conv[-1]["content"] = partial
                yield conv, _build_chatbot_history(conv), ""

            yield conv, _build_chatbot_history(conv), ""

        send_btn.click(
            fn=on_send,
            inputs=[chat_input, conv_state, app_state, chatbot],
            outputs=[conv_state, chatbot, chat_input],
        )
        chat_input.submit(
            fn=on_send,
            inputs=[chat_input, conv_state, app_state, chatbot],
            outputs=[conv_state, chatbot, chat_input],
        )

        # ── Initial page load ─────────────────────────────────────────────── #
        def on_load(state):
            # Model is always ready here because load_llm() runs before app.launch()
            status = "Ready to learn!"
            return (
                gr.update(value=get_breadcrumb(state)),
                gr.update(value=get_nav_title(state)),
                gr.update(visible=False),
                status,
            )

        demo.load(
            fn=on_load,
            inputs=[app_state],
            outputs=[breadcrumb_md, nav_title_md, back_btn, status_md],
        )

    return demo


# ============================================================================
# MODIFIED SECTION — Entry point
# Foreground model loading keeps Colab/Kaggle progress visible.
# In PROD mode the knowledgebase is auto-downloaded before anything else.
# ============================================================================
if __name__ == "__main__":
    # ── Wipe all model-download / pip chatter before Gradio appears ──────
    clear_output_logs()

    print("=" * 60)

    # ── Step 0 (PROD only): auto-download knowledgebase ──────────────────
    if STAGE == "PROD":
        print("Step 0 / 3  —  Preparing knowledgebase ...")
        download_knowledgebase()
        print("  ✓  Knowledgebase ready.\n")

    print("Step 1 / 3  —  Loading sentence embedder ...")
    load_embedder()
    print("  ✓  Embedder ready.\n")

    print("Step 2 / 3  —  Loading LLM (Qwen2.5-7B, 4-bit quantised).")
    print("               This may take 3-5 minutes on first run ...")
    load_llm()
    print("  ✓  LLM ready.\n")

    print("Step 3 / 3  —  Launching Gradio interface ...")
    print("=" * 60)
    app = build_app()
    app.launch(
        server_name="0.0.0.0",
        server_port=7860,
        share=True,
        show_error=True,
    )

Step 0 / 3  —  Preparing knowledgebase ...
Extracting knowledgebase...
Knowledgebase ready.
  ✓  Knowledgebase ready.

Step 1 / 3  —  Loading sentence embedder ...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

  ✓  LLM ready.

Step 3 / 3  —  Launching Gradio interface ...
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://5e2c06143ae79bb2ea.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
